In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_47_try_1.pickle

In [4]:
%%RecordEvent
%%time
### cell 48 ###
# Ensure the question and x-axis title variables are defined in this cell
question_of_interest_cell60 = 'Does your current employer incorporate machine learning methods into their business?'
title_for_x_axis_cell60 = ''

# Combine and compute percentages by year on the GPU
def combine_percentages_by_year(question, x_axis_title, include_2017=False):
    # Prepare year labels and corresponding DataFrames
    years = ['2018', '2019', '2020', '2021', '2022']
    dfs   = [responses_df_2018_cell10,
             responses_df_2019_cell10,
             responses_df_2020,
             responses_df_2021,
             responses_df_2022_cell10]
    if include_2017:
        years.insert(0, '2017')
        dfs.insert(0, responses_df_2017)

    # Concatenate all years into one DataFrame with a 'year' column
    df_all = pd.concat(
        [df[[question]].assign(year=year) for year, df in zip(years, dfs)],
        ignore_index=True
    )

    # Compute counts per (year, answer) and total responses per year
    grouped = df_all.groupby(['year', question]).size().reset_index(name='count')
    totals  = df_all.groupby('year').size().reset_index(name='total')

    # Merge counts with totals and compute percentages
    merged = grouped.merge(totals, on='year')
    merged['percentage'] = (merged['count'] * 100 / merged['total']).round(1)

    # Rename the answer column to the x-axis title (may be empty) and select final columns
    result = (
        merged
        .rename(columns={question: x_axis_title})
        [[x_axis_title, 'percentage', 'year']]
    )
    return result

# Build the combined DataFrame, sort and reset index
maturity_df_combined = (
    combine_percentages_by_year(
        question_of_interest_cell60,
        title_for_x_axis_cell60
    )
    .sort_values(by=['year', 'percentage'], ascending=True)
    .reset_index(drop=True)
)

maturity_df_combined.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0               30 non-null     object
 1   percentage  30 non-null     float64
 2   year        30 non-null     object
dtypes: float64(1), object(2)
memory usage: 2.4+ KB
CPU times: user 219 ms, sys: 7.7 ms, total: 226 ms
Wall time: 226 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_48_try_3.pickle